# v2-nespreso vs ISAS template port — runtime & metrics

Compares **inference runtime** and **validation metrics** (depth RMSE/bias curves and
binned maps) between:

| Stack | Location |
|-------|----------|
| **v2** | `/unity/g2/jmiranda/v2-nespreso` |
| **template port** | `NeSPReSO2_onTemplate/` (this repo) |

Both models load the **same v2 checkpoint weights**. Profile metrics use the **template
train-ready cache** (GoM HDF5, `spatial_pad=0`, `temporal_pad=0`) so inputs/labels are
identical. PCA inverse uses the **template-fitted** models unless noted.

> **Caveat:** v2's native dataset pickle (`config_dataset_full.pkl`) uses a different
> data pipeline (ARGO `.mat` + COAPS). A optional cell times that path separately.

In [ ]:
from __future__ import annotations

import json
import pickle
import sys
import time
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, random_split

# --- paths (edit if needed) ---
V2_REPO = Path("/unity/g2/jmiranda/v2-nespreso")
TEMPLATE_ROOT = Path("..").resolve()          # NeSPReSO2_onTemplate
PROJECT_ROOT = TEMPLATE_ROOT.parent
CACHE_PATH = PROJECT_ROOT / "data/cache/train_ready_c0f62f13ca33.pkl"
V2_CHECKPOINT = Path(
    "/unity/g2/jmiranda/SubsurfaceFields/GEM_SubsurfaceFields/saved_models/"
    "model_Test Loss: 0.8847_2024-10-09 20:45:20_sat.pth"
)
V2_DATASET_PICKLE = Path(
    "/unity/g2/jmiranda/SubsurfaceFields/GEM_SubsurfaceFields/config_dataset_full.pkl"
)

sys.path.insert(0, str(V2_REPO / "src"))
sys.path.insert(0, str(TEMPLATE_ROOT))

from nespreso.data.pickle_compat import load_dataset_pickle
from nespreso.metrics import bias, rmse
from nespreso.models.mlp import PredictionModel as V2PredictionModel
from nespreso.viz.maps import calculate_average_in_bin, plot_bin_map, plot_comparison_maps

from data_loader.data_loaders import NeSPReSODataset, _collate_with_index
from model.loss import sklearn_inverse_transform_pcs
from model.model import PredictionModel as TemplatePredictionModel
from playground import read_json
from preproc.preproc_isas_sat import build_train_cache

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
N_REPEAT = 10          # inference timing repeats
BATCH_SIZE = 512
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
BIN_SIZE = 1.0         # degrees for spatial maps
DEPTH_MAX = 1800       # upper depth for map averaging

print(f"Device: {DEVICE}")
print(f"Template root: {TEMPLATE_ROOT}")
print(f"v2 repo: {V2_REPO}")

In [ ]:
def depth_rmse_bias(residual: np.ndarray, axis: int = 1):
    """Per-depth RMSE and bias (residual = pred - true)."""
    se = residual ** 2
    return np.sqrt(np.mean(se, axis=axis)), np.mean(residual, axis=axis)


def time_inference(model, loader, device, n_repeat, inverse_fn):
    """Return mean seconds per full pass (forward + inverse)."""
    model.eval().to(device)
    t0 = time.perf_counter()
    for _ in range(n_repeat):
        pcs_list = []
        with torch.no_grad():
            for batch in loader:
                x = batch[0].to(device)
                pcs_list.append(model(x).cpu().numpy())
        pcs = np.vstack(pcs_list)
        inverse_fn(pcs)
    elapsed = (time.perf_counter() - t0) / n_repeat
    return elapsed


def make_val_loader(cache, batch_size=512, seed=42):
    inputs = torch.tensor(cache["inputs"], dtype=torch.float32)
    targets = torch.tensor(cache["targets"], dtype=torch.float32)
    ds = NeSPReSODataset(inputs, targets)
    n = len(ds)
    val_len = int(n * VAL_FRAC)
    train_len = int(n * TRAIN_FRAC)
    test_len = n - train_len - val_len
    g = torch.Generator().manual_seed(seed)
    _, val_sub, _ = random_split(ds, [train_len, val_len, test_len], generator=g)
    val_idx = np.array(val_sub.indices)
    loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False, collate_fn=_collate_with_index)
    return loader, val_idx


def pcs_to_profiles(pcs, pca_models, outputs):
    prof = sklearn_inverse_transform_pcs(pcs, pca_models, outputs)
    return prof["temperature"], prof["salinity"]


def align_profiles_to_depths(prof, depth_src_m, depth_tgt_m):
    """Linearly interpolate (n_src, N) profiles onto template depth grid."""
    depth_src_m = np.asarray(depth_src_m, dtype=float).squeeze()
    depth_tgt_m = np.asarray(depth_tgt_m, dtype=float).squeeze()
    if prof.shape[0] == len(depth_tgt_m):
        return prof
    if depth_src_m.size == prof.shape[0]:
        src_x = depth_src_m
    else:
        src_x = np.arange(prof.shape[0], dtype=float)
    out = np.empty((len(depth_tgt_m), prof.shape[1]), dtype=prof.dtype)
    for j in range(prof.shape[1]):
        out[:, j] = np.interp(depth_tgt_m, src_x, prof[:, j])
    return out

## 1. Load shared template cache

In [ ]:
cfg = read_json(TEMPLATE_ROOT / "config.json")

if not CACHE_PATH.exists():
    print("Building cache …")
    CACHE_PATH = Path(build_train_cache(cfg))

t0 = time.perf_counter()
with open(CACHE_PATH, "rb") as f:
    cache = pickle.load(f)
cache_load_s = time.perf_counter() - t0

outputs = OrderedDict(cache["outputs"])
n_comp = list(outputs.values())[0]
input_dim = cache["inputs"].shape[1]
N = cache["inputs"].shape[0]

val_loader, val_idx = make_val_loader(cache, BATCH_SIZE, SEED)
n_val = len(val_idx)

print(f"Cache: {CACHE_PATH.name}  ({cache_load_s*1e3:.1f} ms)")
print(f"N={N}, input_dim={input_dim}, output_dim={cache['targets'].shape[1]}, val={n_val}")

## 2. Load v2 checkpoint into both model classes

In [ ]:
t0 = time.perf_counter()
ckpt = torch.load(V2_CHECKPOINT, map_location="cpu", weights_only=False)
ckpt_load_s = time.perf_counter() - t0

layers = cfg["arch"]["args"]["layers_config"]
dropout = cfg["arch"]["args"]["dropout_prob"]
out_dim = sum(outputs.values())

v2_model = V2PredictionModel(input_dim, layers, out_dim, dropout)
tpl_model = TemplatePredictionModel(input_dim, layers, out_dim, dropout)

v2_model.load_state_dict(ckpt["model_state_dict"])
tpl_model.load_state_dict(ckpt["model_state_dict"])
v2_model.eval()
tpl_model.eval()

pca_v2 = {"temperature": ckpt["pca_temp"], "salinity": ckpt["pca_sal"]}
pca_tpl = cache["pca_models"]

print(f"Checkpoint load: {ckpt_load_s*1e3:.1f} ms")
print(f"v2 input_params: {ckpt['input_params']}")

## 3. Runtime comparison

In [ ]:
def tpl_inverse(pcs):
    return pcs_to_profiles(pcs, pca_tpl, outputs)


def v2_inverse(pcs):
    return pcs_to_profiles(pcs, pca_v2, outputs)


tpl_s = time_inference(tpl_model, val_loader, DEVICE, N_REPEAT, tpl_inverse)
v2_s = time_inference(v2_model, val_loader, DEVICE, N_REPEAT, v2_inverse)

# optional: v2 native dataset load timing
v2_pickle_s = None
if V2_DATASET_PICKLE.exists():
    t0 = time.perf_counter()
    _ = load_dataset_pickle(V2_DATASET_PICKLE)
    v2_pickle_s = time.perf_counter() - t0

runtime_rows = [
    {"step": "template cache load", "impl": "template", "seconds": cache_load_s, "per_profile_us": cache_load_s / N * 1e6},
    {"step": "checkpoint load", "impl": "both", "seconds": ckpt_load_s, "per_profile_us": np.nan},
    {"step": f"inference+inverse (x{N_REPEAT})", "impl": "template port", "seconds": tpl_s, "per_profile_us": tpl_s / n_val * 1e6},
    {"step": f"inference+inverse (x{N_REPEAT})", "impl": "v2 package", "seconds": v2_s, "per_profile_us": v2_s / n_val * 1e6},
]
if v2_pickle_s is not None:
    runtime_rows.append({"step": "v2 dataset pickle load", "impl": "v2 native", "seconds": v2_pickle_s, "per_profile_us": np.nan})

print(f"{'step':40s} {'impl':16s} {'ms':>10s} {'µs/profile':>12s}")
print("-" * 82)
for r in runtime_rows:
    print(f"{r['step']:40s} {r['impl']:16s} {r['seconds']*1e3:10.2f} {r['per_profile_us']:12.1f}")

speedup = v2_s / tpl_s if tpl_s > 0 else float('nan')
print(f"\nTemplate / v2 inference ratio: {speedup:.3f}x  (<1 means template is faster)")

## 4. Prediction agreement (forward pass)

In [ ]:
v2_model.eval().to(DEVICE)
tpl_model.eval().to(DEVICE)

tpl_pcs, v2_pcs = [], []
with torch.no_grad():
    for x, _, _ in val_loader:
        x = x.to(DEVICE)
        tpl_pcs.append(tpl_model(x).cpu().numpy())
        v2_pcs.append(v2_model(x).cpu().numpy())

tpl_pcs = np.vstack(tpl_pcs)
v2_pcs = np.vstack(v2_pcs)
max_pcs_diff = np.max(np.abs(tpl_pcs - v2_pcs))
print(f"Max |Δ PCS| on val set: {max_pcs_diff:.3e}  (expect ≲ 1e-6 with identical weights)")

## 5. Profile metrics — depth RMSE & bias

Ground truth: inverse-PCA of **target** PCs (template PCA).
Predictions: template port vs v2 package (same weights).

In [ ]:
tgt_pcs = cache["targets"][val_idx].astype(np.float64)
true_T, true_S = pcs_to_profiles(tgt_pcs, pca_tpl, outputs)

pred_T_tpl, pred_S_tpl = pcs_to_profiles(tpl_pcs, pca_tpl, outputs)
pred_T_v2, pred_S_v2 = pcs_to_profiles(v2_pcs, pca_tpl, outputs)

res_tpl_T = pred_T_tpl - true_T
res_tpl_S = pred_S_tpl - true_S
res_v2_T = pred_T_v2 - true_T
res_v2_S = pred_S_v2 - true_S

depths_m = np.asarray(cache["PRES"], dtype=float).squeeze()
if depths_m.ndim == 0:
    depths_m = np.arange(true_T.shape[0], dtype=float)
d_mask = depths_m <= DEPTH_MAX
dpt_range = np.where(d_mask)[0].astype(int)

metrics = {
    "template T RMSE": rmse(pred_T_tpl[d_mask], true_T[d_mask]),
    "v2 T RMSE": rmse(pred_T_v2[d_mask], true_T[d_mask]),
    "template S RMSE": rmse(pred_S_tpl[d_mask], true_S[d_mask]),
    "v2 S RMSE": rmse(pred_S_v2[d_mask], true_S[d_mask]),
    "template T bias": bias(pred_T_tpl[d_mask], true_T[d_mask]),
    "v2 T bias": bias(pred_T_v2[d_mask], true_T[d_mask]),
    "template S bias": bias(pred_S_tpl[d_mask], true_S[d_mask]),
    "v2 S bias": bias(pred_S_v2[d_mask], true_S[d_mask]),
}
print(json.dumps(metrics, indent=2))

In [ ]:
rmse_tpl_T, bias_tpl_T = depth_rmse_bias(res_tpl_T, axis=1)
rmse_v2_T, bias_v2_T = depth_rmse_bias(res_v2_T, axis=1)
rmse_tpl_S, bias_tpl_S = depth_rmse_bias(res_tpl_S, axis=1)
rmse_v2_S, bias_v2_S = depth_rmse_bias(res_v2_S, axis=1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Validation depth statistics (0–1800 m)", fontsize=14, fontweight="bold")

for ax, tpl_r, v2_r, title, xlabel in [
    (axes[0, 0], rmse_tpl_T, rmse_v2_T, "Temperature RMSE", "RMSE [°C]"),
    (axes[0, 1], rmse_tpl_S, rmse_v2_S, "Salinity RMSE", "RMSE [PSU]"),
    (axes[1, 0], bias_tpl_T, bias_v2_T, "Temperature bias", "Bias [°C]"),
    (axes[1, 1], bias_tpl_S, bias_v2_S, "Salinity bias", "Bias [PSU]"),
]:
    ax.plot(tpl_r[d_mask], depths_m[d_mask], label="template port", lw=2)
    ax.plot(v2_r[d_mask], depths_m[d_mask], label="v2 package", lw=2, ls="--")
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Depth [m]")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### PCA note: template-fit vs v2 checkpoint PCA

Forward passes are identical, but **profile-space metrics** depend on which PCA is used for inverse transform. The v2 checkpoint PCA was fit on **1801 depth levels** (0–1800 m); the template cache PCA uses **187 ISAS levels** (`DEPH`). The cell below interpolates v2-checkpoint profiles onto the template depth grid before comparing RMSE.

In [ ]:
pred_T_ckpt, pred_S_ckpt = pcs_to_profiles(tpl_pcs, pca_v2, outputs)
# v2 ckpt PCA -> 1801 m levels; interpolate onto template DEPH before RMSE
v2_depth_m = np.arange(pred_T_ckpt.shape[0], dtype=float)
pred_T_ckpt = align_profiles_to_depths(pred_T_ckpt, v2_depth_m, depths_m)
pred_S_ckpt = align_profiles_to_depths(pred_S_ckpt, v2_depth_m, depths_m)

pca_compare = {
    "T RMSE (template PCA)": rmse(pred_T_tpl[d_mask], true_T[d_mask]),
    "T RMSE (v2 ckpt PCA, aligned)": rmse(pred_T_ckpt[d_mask], true_T[d_mask]),
    "S RMSE (template PCA)": rmse(pred_S_tpl[d_mask], true_S[d_mask]),
    "S RMSE (v2 ckpt PCA, aligned)": rmse(pred_S_ckpt[d_mask], true_S[d_mask]),
}
print(json.dumps(pca_compare, indent=2))

## 6. Spatial maps — binned RMSE & bias

Uses v2 `calculate_average_in_bin` / `plot_bin_map` on the GoM validation subset.

In [ ]:
lat_val = cache["LAT"][val_idx]
lon_val = cache["LON"][val_idx]
lat_val = np.floor(lat_val) + BIN_SIZE / 2
lon_val = np.floor(lon_val) + BIN_SIZE / 2

lon_bins = np.arange(np.floor(lon_val.min()) - 0.5, np.ceil(lon_val.max()) + 1.5, BIN_SIZE)
lat_bins = np.arange(np.floor(lat_val.min()) - 0.5, np.ceil(lat_val.max()) + 1.5, BIN_SIZE)
lon_centers = lon_bins + BIN_SIZE / 2
lat_centers = lat_bins + BIN_SIZE / 2

dpt_range = np.where(depths_m <= DEPTH_MAX)[0].astype(int)

grid_rmse_tpl_T, nprof = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_tpl_T, dpt_range, True)
grid_rmse_v2_T, _ = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_v2_T, dpt_range, True)
grid_bias_tpl_T, _ = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_tpl_T, dpt_range, False)
grid_bias_v2_T, _ = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_v2_T, dpt_range, False)

grid_rmse_tpl_S, _ = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_tpl_S, dpt_range, True)
grid_rmse_v2_S, _ = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_v2_S, dpt_range, True)
grid_bias_tpl_S, _ = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_tpl_S, dpt_range, False)
grid_bias_v2_S, _ = calculate_average_in_bin(lon_centers, lat_centers, lon_val, lat_val, res_v2_S, dpt_range, False)

print(f"Map grid: {grid_rmse_tpl_T.shape}, profiles in busiest bin: {nprof.max():.0f}")

In [ ]:
plot_bin_map(lon_bins, lat_bins, grid_rmse_tpl_T, nprof, "Temperature (template)", "RMSE")
plot_bin_map(lon_bins, lat_bins, grid_rmse_v2_T, nprof, "Temperature (v2 package)", "RMSE")
plot_bin_map(lon_bins, lat_bins, grid_bias_tpl_T, nprof, "Temperature (template)", "Bias")
plot_bin_map(lon_bins, lat_bins, grid_bias_v2_T, nprof, "Temperature (v2 package)", "Bias")

In [ ]:
plot_bin_map(lon_bins, lat_bins, grid_rmse_tpl_S, nprof, "Salinity (template)", "RMSE")
plot_bin_map(lon_bins, lat_bins, grid_rmse_v2_S, nprof, "Salinity (v2 package)", "RMSE")
plot_bin_map(lon_bins, lat_bins, grid_bias_tpl_S, nprof, "Salinity (template)", "Bias")
plot_bin_map(lon_bins, lat_bins, grid_bias_v2_S, nprof, "Salinity (v2 package)", "Bias")

In [ ]:
lon_c = lon_centers[:-1]
lat_c = lat_centers[:-1]

plot_comparison_maps(lon_c, lat_c, grid_rmse_tpl_T, grid_rmse_v2_T, "temperature", "v2 package", "RMSE")
plot_comparison_maps(lon_c, lat_c, grid_rmse_tpl_S, grid_rmse_v2_S, "salinity", "v2 package", "RMSE")
plot_comparison_maps(lon_c, lat_c, grid_bias_tpl_T, grid_bias_v2_T, "temperature", "v2 package", "Bias")
plot_comparison_maps(lon_c, lat_c, grid_bias_tpl_S, grid_bias_v2_S, "salinity", "v2 package", "Bias")

## 7. (Optional) v2-native dataset inference timing

Times the full v2 stack on its own validation loader (different data pipeline).

In [ ]:
if not V2_DATASET_PICKLE.exists():
    print("v2 dataset pickle not found — skip native timing.")
else:
    from nespreso.data.splits import split_dataset

    data = load_dataset_pickle(V2_DATASET_PICKLE)
    full_ds = data["full_dataset"]
    if not hasattr(full_ds, "n_components"):
        full_ds.n_components = full_ds.pca_temp.n_components_
    _, val_ds, _ = split_dataset(full_ds, TRAIN_FRAC, VAL_FRAC, 1 - TRAIN_FRAC - VAL_FRAC)
    v2_val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    def v2_native_inverse(pcs):
        full_ds.inverse_transform(pcs)

    native_s = time_inference(v2_model, v2_val_loader, DEVICE, N_REPEAT, v2_native_inverse)
    print(f"v2 native val inference+inverse: {native_s*1e3:.2f} ms  ({native_s/len(val_ds)*1e6:.1f} µs/profile)")
    print(f"Template-cache val inference+inverse: {tpl_s*1e3:.2f} ms  ({tpl_s/n_val*1e6:.1f} µs/profile)")
    print(f"v2 native N={len(full_ds)} val={len(val_ds)} vs template cache N={N} val={n_val}")